# BME 2080 · Module 3 · Week 8
## Data Exploration, Missingness, and Initial Trends

**How to use this notebook:**
- Run each cell in order (Shift+Enter or the ▶ button)
- Look for 🔧 **TRY THIS** markers — change a value, rerun, and observe what changes

---

---
## Overview: Problem Framing and Biological Context

### What is this dataset?
This dataset comes from the **[Cancer Dependency Map](https://depmap.org/portal/) (DepMap)** project at the Broad Institute. It is a **subset** of this database and contains measurements from **212 cancer cell lines** cultured from patient-derived tumors. Each cell line has been characterized across multiple biological layers (called **omics** layers) and each has been treated with 15 different cancer drugs.

### What are we predicting?
Our goal is to predict **drug response** — whether a given cell line will be **sensitive** or **resistant** to a given drug. Drug response is measured as **AUC** (Area Under the dose-response Curve):
- **AUC close to 0** → the drug killed most cells → the cell line is **sensitive**
- **AUC close to 1** → the drug had no effect → the cell line is **resistant**

### Why does this matter clinically?
In precision oncology, the goal is to match the *right drug* to the *right patient*. If we can predict which molecular features predict drug sensitivity, we can use a patient's tumor profile to choose the most likely effective treatment.

### The four data layers in this dataset:
| Prefix | Type | Units | What it measures | Features |
|--------|------|-------|-----------------|----------|
| `exp_` | RNA Expression | log2(TPM+1) | How actively each gene is transcribed | 91 genes |
| `prot_` | Protein Expression | Unitless ratio | Abundance of each protein | 30 proteins |
| `mut_` | Mutation | Binary (0/1) | Whether a cancer gene is mutated (0/1) | 15 genes |
| `drug_` | Drug Response | Unitless (0-1) | AUC per drug (0=sensitive, 1=resistant) | 15 drugs |

### Signaling pathways
This dataset includes genes, proteins, and drugs that are grouped by signaling pathways. For a full list, see the dataset documentation file. The primary genes, proteins, and drugs for each pathway are shown below.

| Pathway | Select Genes | Select Proteins | Drugs |
|---------|--------------|-----------------|-------|
| EGFR/RAS/MAPK | EGFR, ERBB2, MAP2K1, KRAS, BRAF | ERBB2, BRAF, MAPK1 | Gefitinib, Erlotinib, Dabrafenib |
| PI3K/AKT/mTOR | PIK3CA, PTEN, MTOR | PIK3CA, PTEN, MTOR, RPTOR | Alpelisib, Buparlisib, Everolimus |
| JAK/STAT | JAK1, JAK2, STAT3 | JAK2, STAT3, SOCS3 | Ruxolitinib, Baracitinib, Napabucasin |
| TGF-β/SMAD | TGFB1, TGFBR1, SMAD3 | SMAD3, SMAD4 | Repsox |
| EMT & SRC Signaling | CDH1, CDH2, SNAI1, SRC | CDH1, CDH2, SNAI1, ITGA2 | KX2-391 |

---
## Section 1: Setup and Data Loading

Before any analysis, we load our Module 3 Python libraries and the dataset.
Each library has a specific role:
- **pandas** — loading, filtering, and summarizing tabular data
- **numpy** — numerical operations
- **scipy** — scientific and computing functionalities, statistics
- **matplotlib / seaborn** — data visualization
- **scikit-learn** — machine learning (we won't import this until Weeks 9 and 10)

In [ ]:
# ── Block 1: Import libraries ──────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set a consistent visual style for all plots
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('✅ Libraries loaded successfully!')

In [ ]:
# ── Block 2: Load the dataset ─────────────────────────────────
# The data is hosted on the course GitHub page, so you don't have to directly upload

DATA_URL = 'https://raw.githubusercontent.com/cr546-collab/bme2080-module3/main/teaching_multiomics_AUC_212lines.csv'

df = pd.read_csv(DATA_URL)

print(f'✅ Dataset loaded!')

In [ ]:
# ── Block 3: Determine the dimensions of the dataset and the type of the data ─────────────────────────────────

# Important data types in Python include integers (int), numbers with decimals
# (float), mixed or textual data (object or string), or true/false data (boolean)

print(f'   Rows (cell lines): {df.shape[0]}')
print(f'   Columns: {df.shape[1]}')

df.info()

In [ ]:
# ── Block 4: Organize columns by data type ────────────────────────────────────────

# Here, we group columns by prefix (using list comprehension), which allows you
# to explore and manipulate one type of data/feature at a time if desired.

meta_cols  = ['DepMap_ID', 'disease']
exp_cols   = [c for c in df.columns if c.startswith('exp_')]
prot_cols  = [c for c in df.columns if c.startswith('prot_')]
mut_cols   = [c for c in df.columns if c.startswith('mut_')]
drug_cols  = [c for c in df.columns if c.startswith('drug_')]

# This code gives you the number of columns of each type.

print('Feature inventory:')
print(f'  Metadata:           {len(meta_cols)} columns  → {meta_cols}')
print(f'  RNA expression:     {len(exp_cols)} genes')
print(f'  Protein expression: {len(prot_cols)} proteins')
print(f'  Mutations:          {len(mut_cols)} genes')
print(f'  Drug response:      {len(drug_cols)} drugs')
print()
print('Drugs in the dataset:')
for d in sorted(drug_cols):
    print(f'  {d.replace("drug_", "")}'  )

In [ ]:
# ── Block 5: How many cell lines per cancer type? ──────────────────────────────────

cancer_counts = df['disease'].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(
    cancer_counts.index,
    cancer_counts.values,
    color='steelblue',
    edgecolor='white',
    linewidth=0.8
)
ax.bar_label(bars, padding=4, fontsize=11)
ax.set_xlabel('Number of Cell Lines')
ax.set_title('Cell Lines per Cancer Type', fontweight='bold')
plt.tight_layout()
plt.show()

print(cancer_counts)


---
## Section 2: Initial Data Inspection

Look at the actual values before doing any analysis to explore the data structure, type, etc.
This step also catches unexpected issues, including wrong data types, out-of-range values, columns accidentally loaded as strings, etc.

In [ ]:
# ── Block 6: Preview the first few rows ───────────────────────────────────────────
# .head(n) shows the first n rows to familiarize yourself with data structure
# Click the "table" icon to the far right (scroll) to see more of the data

df.head(5)

In [ ]:
# ── Block 7: Summary statistics for drug response columns ─────────────────────────
# .describe() gives count, mean, std, min, max, and quartiles

print('Drug Response AUC — Summary Statistics')
print('(Remember: 0 = sensitive, 1 = resistant)')
print()
df[drug_cols].describe().round(3)

In [ ]:
# Block 8: More summary statistics
# 🔧 TRY THIS ─────────────────────────────────────────────────────────────

# Change the variable below to explore statistics for different data types.
# Options: exp_cols, prot_cols, mut_cols, drug_cols


# >>> CHANGE THIS LINE <<<
columns_to_inspect =mut_cols

df[columns_to_inspect].describe().round(3)

---
## Section 3: Missing Data

Real biological datasets almost always have missing values. Before building
any model, we need to understand *where* data is missing, *how much*,
and *why*.

### Types of missingness

| Type | Definition | Example in our data |
|------|-----------|---------------------|
| **MCAR** | Missing Completely At Random | A sample was accidentally dropped |
| **MAR** | Missing At Random — related to *other observed* variables | A drug was not tested in a certain cancer type |
| **MNAR** | Missing Not At Random — related to the *missing value itself* | A drug was skipped because the cell line was already known to be resistant |



In [ ]:
# ── Block 9: Count missing values across the whole dataset ─────────────────────────

missing_counts = df.isnull().sum()
missing_pct    = (missing_counts / len(df) * 100).round(1)

# Show only columns that actually have missing values

missing_summary = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing %':     missing_pct
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

print(f'Columns with ANY missing values: {len(missing_summary)} out of {len(df.columns)}')
print()
print(missing_summary.to_string())

In [ ]:
# ── Block 10: Visualize missingness in drug response columns ────────────────────────

drug_missing = df[drug_cols].isnull().sum().sort_values(ascending=True)
drug_names   = [c.replace('drug_', '') for c in drug_missing.index]

fig, ax = plt.subplots(figsize=(10, 6))

# Color bars: red if missing, green if complete

colors = ['#ff6b6b' if v > 0 else '#39d98a' for v in drug_missing.values]
bars   = ax.barh(drug_names, drug_missing.values,
                 color=colors, edgecolor='white')
ax.bar_label(bars, padding=3)
ax.set_xlabel('Number of Missing Values (out of 212 cell lines)')
ax.set_title('Missing Drug Response Values per Drug', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Most missing:  {drug_missing.index[-1].replace("drug_", "")} '
      f'({drug_missing.iloc[-1]} missing, {drug_missing.iloc[-1]/212*100:.1f}%)')
print(f'Least missing: {drug_missing.index[0].replace("drug_", "")} '
      f'({drug_missing.iloc[0]} missing)'  )

In [ ]:
# ── Block 11: Test for MAR: is missingness related to cancer type? ─────────────────

# If missingness is evenly spread across cancer types → suggests MCAR
# If certain cancer types have more missing → suggests MAR

# 🔧 TRY THIS ─────────────────────────────────────────────────────────────
# Change the drug below and see if the missing pattern differs.
# Try: 'drug_Buparlisib', 'drug_Baricitinib', 'drug_Everolimus', 'drug_Trametinib'
# Trametinib has ZERO missing — how does that compare to the others?

# >>> CHANGE THIS LINE <<<
drug_to_check = 'drug_Everolimus'

missing_by_cancer = (df.groupby('disease')[drug_to_check]
                       .apply(lambda x: x.isnull().sum())
                       .sort_values(ascending=False))

total_by_cancer   = df['disease'].value_counts()
missing_pct_by_c  = (missing_by_cancer / total_by_cancer * 100).round(1)

result = pd.DataFrame({
    'Total Cell Lines': total_by_cancer,
    'Missing':          missing_by_cancer,
    'Missing %':        missing_pct_by_c
}).sort_values('Missing %', ascending=False)

print(f'Missing {drug_to_check.replace("drug_", "")} by cancer type:')
print(result.to_string())
print()
print('Is missingness evenly distributed across cancer types?')
print('If not, this suggests MAR rather than MCAR.'  )

---
## Section 4: Exploratory Distributions

Now let's explore the distributions of key features.
Knowing the shape, range, and skew of your data is essential before modeling. Outliers, non-normality, and scale differences all affect model performance.

In [ ]:
# ── Block 12: AUC distributions for all 15 drugs ───────────────────────────────────

# Each histogram shows how sensitive vs resistant the cell lines are for one drug
# Distribution shifted LEFT  → more sensitive cell lines
# Distribution shifted RIGHT → most cell lines are resistant

fig, axes = plt.subplots(3, 5, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(sorted(drug_cols)):
    drug_name = col.replace('drug_', '')
    axes[i].hist(df[col].dropna(), bins=20,
                 color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(drug_name, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('AUC')
    axes[i].set_xlim(0, 1.05)
    # Red dashed line at the sensitivity threshold
    axes[i].axvline(0.7, color='red', linestyle='--',
                    linewidth=1.2, label='AUC = 0.7')

axes[0].legend(fontsize=8)
plt.suptitle(
    'AUC Distributions — 15 Drugs\n'
    'Red line = AUC 0.7 (left of line = sensitive)',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
# Block 13: 🔧 TRY THIS ─────────────────────────────────────────────────────────────

# Change the threshold below and observe how the number of "sensitive"
# cell lines changes for each drug.

# This is one of the most important decisions in drug response modeling:
# where do you draw the line between sensitive and resistant?
# While AUC = 0.7 is a common threshold, there is no single correct answer. It is a biological and statistical judgment.

# Try: 0.5, 0.7, 0.8, 0.9

# >>> CHANGE THIS LINE <<<
sensitivity_threshold = 0.7

print(f'Percentage of cell lines SENSITIVE (AUC < {sensitivity_threshold}) per drug:')
print(f'Sorted from most to least sensitive.\n')

rates = {}
for col in drug_cols:
    vals = df[col].dropna()
    rates[col.replace('drug_', '')] = (vals < sensitivity_threshold).mean() * 100

rate_series = pd.Series(rates).sort_values(ascending=False)
for drug, pct in rate_series.items():
    bar = '█' * int(pct / 2)
    print(f'  {drug:<15} {pct:>5.1f}%  {bar}')

print()
print(f'At threshold = {sensitivity_threshold}, most drugs have very few sensitive cell lines.')
print('This is called CLASS IMBALANCE — a major challenge in drug response modeling.'  )

In [ ]:
# ── Block 14: RNA expression distributions for selected genes ───────────────────────

# Values are log2(TPM + 1) — higher = more expression
# Most genes follow a roughly normal distribution

genes_to_plot = [
    'exp_EGFR', 'exp_ERBB2', 'exp_KRAS',  'exp_BRAF',
    'exp_PIK3CA', 'exp_PTEN', 'exp_JAK2', 'exp_SRC'
]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, gene in enumerate(genes_to_plot):
    if gene in df.columns:
        axes[i].hist(df[gene].dropna(), bins=25,
                     color='#39d98a', edgecolor='white', alpha=0.85)
        axes[i].set_title(gene.replace('exp_', ''), fontsize=11, fontweight='bold')
        axes[i].set_xlabel('log2(TPM + 1)')
        axes[i].axvline(df[gene].mean(), color='red', lw=1.5,
                        linestyle='--', label=f"mean={df[gene].mean():.2f}")
        axes[i].legend(fontsize=8)
    else:
        axes[i].text(0.5, 0.5, f'{gene}\nnot found',
                     ha='center', va='center', transform=axes[i].transAxes)

plt.suptitle('RNA Expression Distributions (selected genes)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Block 15: Mutation prevalence ───────────────────────────────────────────────────

# Mutations are binary: 0 = wild-type, 1 = mutated
# This shows what fraction of the 212 cell lines carry each mutation

mut_rates  = df[mut_cols].mean() * 100
mut_names  = [c.replace('mut_', '') for c in mut_rates.index]
mut_rates.index = mut_names
mut_rates  = mut_rates.sort_values(ascending=True)

# Color by rate: common (red), moderate (yellow), rare (purple)
colors = ['#ff6b6b' if v > 10
          else '#ffd166' if v > 5
          else '#7b5cf0'
          for v in mut_rates.values]

fig, ax = plt.subplots(figsize=(10, 8))   # taller figure for 15 genes
bars = ax.barh(mut_rates.index, mut_rates.values,
               color=colors, edgecolor='white', height=0.6)
ax.bar_label(bars, fmt='%.1f%%', padding=4, fontsize=10)
ax.set_xlabel('% of Cell Lines with Mutation', fontsize=11)
ax.set_title('Mutation Prevalence across 212 Cell Lines', fontweight='bold', pad=12)
ax.set_xlim(0, mut_rates.max() * 1.25)   # extra space for labels
ax.tick_params(axis='y', labelsize=10)
plt.tight_layout()
plt.show()


---
## Section 5: Initial Correlations and Trends

Now we start asking: are there features that are already obviously predictive
of drug response? This is the bridge between data exploration and modeling.

A **negative** correlation between a feature and AUC means:
higher feature value → lower AUC → more sensitive to the drug.

A **positive** correlation means: higher feature value → more resistant.

In [ ]:
# ── Block 16: Compute correlation of all features with all drugs ────────────────────

feature_pool = exp_cols + prot_cols  # continuous features only
corr_matrix  = pd.DataFrame(index=feature_pool, columns=drug_cols, dtype=float)

# Calculating Pearson correlation coefficient (r)
# -1 indicates perfect negative linear correlation
# +1 indicates perfect positive linear correlation

for feat in feature_pool:
    for drug in drug_cols:
        valid = df[[feat, drug]].dropna()
        if len(valid) > 10:
            r, _ = stats.pearsonr(valid[feat], valid[drug])
            corr_matrix.loc[feat, drug] = r

print(f'Correlation matrix computed: {corr_matrix.shape[0]} features x {corr_matrix.shape[1]} drugs')
print(f'Strongest positive correlation: {corr_matrix.max().max():.3f}')
print(f'Strongest negative correlation: {corr_matrix.min().min():.3f}')

In [ ]:
# ── Block 17: Heatmap: top features correlated with drug response ───────────────────

# Shows which features are most associated with drug sensitivity/resistance
# across all 15 drugs simultaneously

# 🔧 TRY THIS ─────────────────────────────────────────────────────────────
# Change n_top_features to show more or fewer rows.
# Try: 10, 20, 30

# >>> CHANGE THIS LINE <<<
n_top_features = 20

# Select features with the highest max absolute correlation across any drug
max_abs_corr  = corr_matrix.abs().max(axis=1)
top_features  = max_abs_corr.nlargest(n_top_features).index

# Shorten names for display
display_corr = corr_matrix.loc[top_features].copy()
display_corr.index   = [f.replace('exp_',  'RNA: ')
                          .replace('prot_', 'Prot: ')

                         for f in display_corr.index]
display_corr.columns = [c.replace('drug_', '') for c in display_corr.columns]

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(
    display_corr.astype(float),
    cmap='RdBu_r', center=0, vmin=-0.6, vmax=0.6,
    ax=ax, linewidths=0.3,
    annot=True, fmt='.2f', annot_kws={'size': 7},
    cbar_kws={'label': 'Pearson r'}
)
ax.set_title(
    f'Top {n_top_features} Features Most Correlated with Drug Response\n'
    'Blue = negative r (higher feature → more sensitive)  |  '
    'Red = positive r (higher feature → more resistant)',
    fontweight='bold', fontsize=11
)
ax.set_xlabel('Drug')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

---
**Drugs and Targets**

In the code block below, explore correlations between drug response and the expression of the target. All drugs and their targets are shown below:

| Drug | Target(s) | Pathway |
|------|-----------|---------|
| Gefitinib | EGFR | EGFR/RAS/MAPK |
| Erlotinib | EGFR | EGFR/RAS/MAPK |
| Afatinib | EGFR, HER2, HER4 | EGFR/RAS/MAPK |
| Neratinib | EGFR, HER2, HER4 | EGFR/RAS/MAPK |
| Vemurafenib | BRAF V600E | EGFR/RAS/MAPK |
| Dabrafenib | BRAF V600E | EGFR/RAS/MAPK |
| Trametinib | MEK1/2 | EGFR/RAS/MAPK |
| Alpelisib | PI3Kα | PI3K/AKT/mTOR |
| Buparlisib | Pan-PI3K | PI3K/AKT/mTOR |
| Everolimus | mTORC1 | PI3K/AKT/mTOR |
| Ruxolitinib | JAK1/2/3 | JAK/STAT |
| Baricitinib | JAK1/2 | JAK/STAT |
| Napabucasin | STAT3 | JAK/STAT |
| Repsox | TGFBR1 | TGF-β/SMAD |
| KX2-391 | SRC | EMT/SRC |

In [ ]:
# ── Block 18: Explore feature-drug relationships ────────────


cancer_colors = {
    'Breast':                        '#ff6b9d',
    'Colorectal':                    '#39d98a',
    'Lung Adenocarcinoma':           '#00e5ff',
    'Lung Squamous Cell Carcinoma':  '#7b5cf0',
    'Ovary':                         '#ffd166',
    'Pancreas':                      '#ff9f43',
    'Skin':                          '#ff6b6b'
}

# 🔧 TRY THIS ─────────────────────────────────────────────────────────────
# Change x_feature and y_drug to explore different biological relationships.
#
# Suggested combinations:
#   exp_EGFR   + drug_Gefitinib    → EGFR inhibitor: does expression predict response?
#   exp_BRAF   + drug_Vemurafenib  → BRAF inhibitor
#   exp_BRAF   + drug_Trametinib   → MEK inhibitor (downstream of BRAF)
#   exp_PIK3CA + drug_Alpelisib    → PI3K inhibitor
#   exp_SRC    + drug_Kx2391       → SRC inhibitor
#   prot_ERBB2 + drug_Afatinib     → pan-HER inhibitor
#
# Or pick another combination based on the table above and the protein list.

# >>> CHANGE THESE TWO LINES <<<
x_feature = 'exp_PIK3CA'
y_drug     = 'drug_Alpelisib'

plot_data = df[[x_feature, y_drug, 'disease']].dropna()
r, p      = stats.pearsonr(plot_data[x_feature], plot_data[y_drug])

fig, ax = plt.subplots(figsize=(9, 6))

for cancer, group in plot_data.groupby('disease'):
    ax.scatter(
        group[x_feature], group[y_drug],
        color=cancer_colors.get(cancer, 'gray'),
        alpha=0.75, edgecolors='white', linewidths=0.5,
        s=55, label=cancer
    )

# Regression line
m, b    = np.polyfit(plot_data[x_feature], plot_data[y_drug], 1)
x_line  = np.linspace(plot_data[x_feature].min(), plot_data[x_feature].max(), 100)
ax.plot(x_line, m * x_line + b, 'k--', linewidth=1.5, alpha=0.6, label='Trend')

drug_label = y_drug.replace('drug_', '')
ax.set_xlabel(x_feature, fontsize=12)
ax.set_ylabel(f'{drug_label} AUC  (lower = more sensitive)', fontsize=12)
ax.set_title(
    f'{x_feature} vs {drug_label}\n'
    f'Pearson r = {r:.3f}  (p = {p:.4f})',
    fontweight='bold'
)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Interpret the result
if p < 0.05:
    direction = 'negative' if r < 0 else 'positive'
    print(f'✅ Statistically significant {direction} correlation (p < 0.05)')
    if r < 0:
        print(f'   Higher {x_feature} → Lower AUC → More sensitive to {drug_label}')
    else:
        print(f'   Higher {x_feature} → Higher AUC → More resistant to {drug_label}')
else:
    print(f'❌ No statistically significant correlation (p = {p:.4f})')

---
## Week 8 Summary

You have now completed an exploratory analysis of the multi-omics dataset.

| Step | Key takeaway |
|------|-------------|
| **Data loading** | 212 cell lines, 7 cancer types, 160 features across 4 omics layers |
| **Missingness** | Drug response values are missing in a non-random pattern (MAR/MNAR) |
| **Distributions** | Most cell lines are resistant to most drugs; expression varies meaningfully across genes and cancer types |
| **Correlations** | Some features show significant correlations with drug response, but individual correlations are weak (r ≈ 0.1–0.4) |


**Next week:** We will build our first machine learning models: Elastic Net and Random Forest.

---
*BME 2080 · Module 3 · Spring 2026*